In [1]:
import os
import cv2
import glob
import json
import pickle
import random
import numpy as np
from tqdm import tqdm

In [2]:
LABELS = [
    "R1", "R2", "R3", "R4", # crops
    "RM1", "RM2",
    "RSL1", "RSM1",
    "LTE", "MTE", # edges
    "LTL", "MTM",
    "LTC", "MTC", # center
    "LTM", "MTL"
]

# LABELS = [
#     "R1", "R2", "R3", "R4", # crops
#     "RM1", "RM2",
#     "RSL1", "RSM1",
#     "RSLT1", "RSMT1"
# ]

def get_keypoints(filepath):
    keypoints = {}

    with open(filepath, "r") as f:
        text = f.read()
    f.close()

    for line in text.split("\n"):
        if any(line.startswith(l) for l in LABELS):
            items = line.split()
            keypoints[items[0]] = (float(items[1]), float(items[3]))
    
    return keypoints

In [3]:
# SKIP LIST
ankle_list = [
    "dataset/2019_10월/1002 한순희_KP(RT불가)",
    "dataset/2020_01월/1221 박순자_KP_LT(x-ray잘림)_불가",
    "dataset/2018_11월/1114 조종순(불가)"
]

In [5]:
folderpaths = glob.glob("/dataset/*")

In [4]:
folderpaths = glob.glob("dataset/*/*")
#folderpaths = glob.glob("dataset/2018_3월/*")
data = []

for path in folderpaths:
    if path in ankle_list:
        print(path)
        continue
    jpg_files = glob.glob(os.path.join(path, "M2" , "*.jpg"))
    txt_files = glob.glob(os.path.join(path, "M2" , "*.txt"))
    #dicom_filepath = glob.glob(os.path.join(path, '001'))
    
    if len(jpg_files) > 1:
        print(f"Found multiple jpg files -> {jpg_files}")
    if len(txt_files) > 1:
        print(f"Found multiple txt files: {path}")
    
    try:
        #ds = pydicom.dcmread(dicom_filepath[0])
        #orientation = ds.get("PatientOrientation", None)
        #print(orientation)
        #spacing = ds.get("PixelSpacing", None)
        #orientation = ds.get("PatientOrientation", None)
        #print(orientation)
        img_path = jpg_files[0]
        keypoints = get_keypoints(txt_files[0])

        for l in LABELS:
            if l not in keypoints:
                print(f"{l} is missing: {path}")
        
        data.append((img_path, keypoints))

    except:
        print(f"Something wrong: {path}")

print(f"data size: {len(data)}")

dataset/2018_11월/1114 조종순(불가)
dataset/2019_10월/1002 한순희_KP(RT불가)
dataset/2020_01월/1221 박순자_KP_LT(x-ray잘림)_불가
data size: 3205


In [5]:
# #random.shuffle(data)
# total = len(data)
# train_end = int(total * 0.8)
# val_end = int(total * 0.9)

# train_data = data[:train_end]
# val_data = data[train_end:val_end]
# test_data = data[val_end:]

# with open("pickle/train.pkl", "wb") as f:
#     pickle.dump(train_data, f)

# with open("pickle/val.pkl", "wb") as f:
#     pickle.dump(val_data, f)

# with open("pickle/test.pkl", "wb") as f:
#     pickle.dump(test_data, f)

In [5]:
random.shuffle(data)

In [6]:
fold_size = len(data) // 5
print(f"fold_size: {fold_size}")
folds = [data[i*fold_size:(i+1)*fold_size] for i in range(5)]

with open("pickle/folds.pkl", "wb") as f:
    pickle.dump(folds, f)

fold_size: 641
